In [1]:
import warnings
warnings.filterwarnings('ignore')

(Cookbook_Saving_halogen_bonds)=
# Saving halogen-bond analyses

*Calculating, naming and persisting directional halogen observations.*

Keep calculated observations with their source system and preserve geometry, empty frames and scientific attribution when extracting a subset. This controlled donor/carbonyl example demonstrates storage rather than biological binding.

:::{versionadded} 1.0.0
:::

## Preparing structures

In [2]:
import molsysmt as msm
from rdkit import Chem
import numpy as np
from molsysmt import pyunitwizard as puw

molsys = msm.convert(Chem.MolFromSmiles('CCl.C=O'), to_form='molsysmt.MolSys')
xyz = np.array([[-.15, 0, 0], [0, 0, 0], [.36, .12*np.sqrt(.75), 0], [.30, 0, 0]])
coordinates = np.repeat(xyz[None], 3, axis=0)
coordinates[1, [2, 3]] += [0, .5, 0]
molsys.structures.append(coordinates=puw.quantity(coordinates, 'nm'))

## Calculating and attaching

Calculation returns an independent result. The name is chosen by the caller; attachment declares matching atom and structure indices.

In [3]:
analysis = msm.interactions.halogen_bonds.get_halogen_bonds(molsys, pbc=False)
molsys.interactions = {**molsys.interactions, 'halogens': analysis}
assert analysis.evaluated_structure_indices.tolist() == [0, 1, 2]
assert analysis.query(structure_indices=[1]).n_interactions == 0
print(analysis.n_interactions, analysis.software)

2 {'molsysmt': '0.21.0+606.ga03eb4bf6', 'rdkit': '2025.09.5'}


## Saving and loading

Public conversion writes H5MSM 0.5. Loading retains original producer versions, method parameters, four-role identities, occurrence indices and bibliography.

In [4]:
from tempfile import TemporaryDirectory
from pathlib import Path
with TemporaryDirectory() as directory:
    path = str(Path(directory) / 'system.h5msm')
    msm.convert(molsys, to_form=path)
    loaded = msm.convert(path, to_form='molsysmt.MolSys')
restored = loaded.interactions['halogens']
assert restored.parameters == analysis.parameters
assert restored.software == analysis.software
np.testing.assert_array_equal(restored.to_dict()['occurrence_indices'], analysis.to_dict()['occurrence_indices'])
print(restored.evaluated_structure_indices)

[0 1 2]


## Extracting and reordering

Extraction remaps all present domains. Removing the acceptor reference atom removes the four-role relation without recalculating the remaining observations.

In [5]:
subset = msm.extract(loaded, selection=[3, 2, 1, 0], structure_indices=[2, 0, 1])
assert subset.interactions['halogens'].n_interactions == 2
print(subset.interactions['halogens'].atom_source_indices)
print(subset.interactions['halogens'].structure_source_indices)
without_reference = msm.extract(loaded, selection=[0, 1, 3])
assert without_reference.interactions['halogens'].n_interactions == 0
print(without_reference.interactions['halogens'].evaluated_structure_indices)

[0 1 2 3]
[2 0 1]
[0 1 2]


:::{seealso}
:class: dropdown

- {ref}`Tutorial_Get_halogen_bonds` — detector and periodic geometry.
- {ref}`user-tools-interactions-result` — sparse queries and declared scope.
- {ref}`user-foundations-native-world-classes-molsysmt-molsys` — named analysis attachment.
:::